# Fine-Tune `vllm-sr/d3-edge` on Indonesian Legal Navigation Trajectories

This Google Colab notebook fine-tunes the 0.6B System 1 decision model `vllm-sr/d3-edge` on statutory tree navigation decisions (`root_to_bab` and `bab_to_pasal`) from Indonesian legal documents.

- **Dataset**: [`ersaasyhar/d3-indo-legal-navigation`](https://huggingface.co/datasets/ersaasyhar/d3-indo-legal-navigation)
- **Base Model**: [`vllm-sr/d3-edge`](https://huggingface.co/vllm-sr/d3-edge) (Qwen3.5 backbone + 255-way readout)
- **Target Model**: [`ersaasyhar/d3-edge-indo-legal`](https://huggingface.co/ersaasyhar/d3-edge-indo-legal)
- **Optimization**: 4-Epoch Elbow Method with automated knee-point checkpoint selection and trajectory visualization.
- **Hardware**: Google Colab T4 GPU (free tier) or A100/L4 GPU

In [2]:
# Cell 1: Environment Setup (Kaggle GPU Optimized)
# Do NOT reinstall torch/torchvision: Kaggle already provides pre-configured CUDA PyTorch.
# flash-linear-attention provides fused Triton kernels for Gated DeltaNet layers.
import os

# Must be set BEFORE torch touches the GPU, otherwise it is ignored.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip uninstall -y -q torchao
!pip install -q peft transformers huggingface_hub safetensors einops matplotlib tqdm
!pip install -q flash-linear-attention

import importlib.util

import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Count: {torch.cuda.device_count()}")
    print(f"Device 0: {torch.cuda.get_device_name(0)}")
    print(
        f"Compute capability: {torch.cuda.get_device_capability(0)}  (>= 8.0 needed for fast bf16)"
    )

# Verify fast kernels are available (the Gated DeltaNet layers are VERY slow without them)
if importlib.util.find_spec("fla") is not None:
    print("flash-linear-attention detected! (Fast Gated DeltaNet Triton kernels active)")
else:
    print(
        "WARNING: flash-linear-attention NOT found -> slow reference implementation will be used."
    )
    print("         Fix the pip install above (check Internet is ON) and restart the session.")

if importlib.util.find_spec("causal_conv1d") is None:
    print(
        "Note: causal-conv1d not installed (optional; torch fallback conv is used, slightly slower)."
    )

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 122.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 114.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 5.9 MB/s eta 0:00:00
PyTorch Version: 2.14.1+cu130
CUDA Available: True
Device: Tesla T4


In [3]:
# Cell 2: Authentication
import os

from huggingface_hub import HfApi, login

try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    from getpass import getpass

    hf_token = getpass("Enter your Hugging Face write token: ")

login(token=hf_token, add_to_git_credential=True)
api = HfApi()
print(f"Authenticated as: {api.whoami()['name']}")

Authenticated as: ersaasyhar


In [4]:
# Cell 3: Dataset Loading (Local Upload or Hugging Face Hub)
import json
import os
import random
from pathlib import Path

from huggingface_hub import hf_hub_download

# Prioritize local uploaded dataset file, fallback to Hugging Face Hub download
local_candidates = [
    Path("d3_navigation_train.jsonl"),
    Path("train.jsonl"),
    Path("data/d3_navigation_train.jsonl"),
]
dataset_file = next((p for p in local_candidates if p.exists()), None)

if dataset_file:
    print(f"Loading local dataset from: {dataset_file}")
    with open(dataset_file, "r", encoding="utf-8") as f:
        records = [json.loads(line) for line in f if line.strip()]
else:
    print("Downloading dataset from Hugging Face Hub: ersaasyhar/d3-indo-legal-navigation...")
    dataset_path = hf_hub_download(
        repo_id="ersaasyhar/d3-indo-legal-navigation",
        filename="train.jsonl",
        repo_type="dataset",
    )
    with open(dataset_path, "r", encoding="utf-8") as f:
        records = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(records)} trajectory records.")
# Train / Validation Split (90/10)
random.seed(42)
shuffled = records.copy()
random.shuffle(shuffled)
val_size = min(200, max(40, int(len(records) * 0.1)))
train_set = shuffled[val_size:]
val_set = shuffled[:val_size]
print(f"Train samples: {len(train_set)}, Validation samples: {len(val_set)}")

train.jsonl:   0%|          | 0.00/699k [00:00<?, ?B/s]

Loaded 404 trajectory records from Hugging Face Hub.
Train samples: 364, Validation samples: 40


In [5]:
# Cell 4: Model Initialization & PEFT LoRA Setup
import peft.import_utils
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModel

# Prevent PEFT from failing on environments with incompatible pre-installed torchao
peft.import_utils.is_torchao_available = lambda: False
try:
    import peft.tuners.lora.torchao

    peft.tuners.lora.torchao.is_torchao_available = lambda: False
except Exception:
    pass

BASE_MODEL_NAME = "vllm-sr/d3-edge"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# The d3 remote code REJECTS any dtype except bf16 / None / "auto" inside from_pretrained
# (passing float16 raises ValueError). So: load it exactly as shipped, then cast the backbone ourselves.
print(f"Loading base decision model {BASE_MODEL_NAME} (checkpoint-native BF16) on {device}...")
model = AutoModel.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
model = model.to(device)

backbone = model.runtime.backbone.to(device)
readout = model.runtime.readout.to(device)  # FP32 by design; keep it FP32


@torch.no_grad()
def _probe_hidden(bb, samples):
    """Last-token hidden state for a few real samples (one at a time, no padding)."""
    bb.eval()
    outs = []
    for s in samples:
        q = {"type": "choice", "instructions": s["instructions"], "criteria": s["criteria"]}
        text = model.runtime.text(s["state"], q)
        ids = model.runtime.tokenizer([text], add_special_tokens=False)["input_ids"][0][-256:]
        x = torch.tensor([ids], device=device)
        h = bb(input_ids=x, attention_mask=torch.ones_like(x), use_cache=False).last_hidden_state[
            :, -1
        ]
        outs.append(h.float().cpu())
    return torch.cat(outs)


# T4 (compute capability 7.5) has fast FP16 tensor cores but only *emulated* BF16 (several times slower).
# Cast the backbone to FP16 BEFORE adding LoRA (so adapters stay FP32), and verify numerics first.
model_dtype = next(backbone.parameters()).dtype
want_fp16 = device.type == "cuda" and torch.cuda.get_device_capability(0)[0] < 8
if want_fp16:
    probe_samples = val_set[:8]
    ref = _probe_hidden(backbone, probe_samples)  # bf16 reference
    backbone.to(torch.float16)
    test = _probe_hidden(backbone, probe_samples)
    cos = torch.nn.functional.cosine_similarity(ref, test, dim=-1)
    finite = bool(torch.isfinite(test).all())
    print(f"fp16 vs bf16 probe: finite={finite}, min cosine similarity={cos.min().item():.4f}")
    if finite and cos.min().item() > 0.98:
        model_dtype = torch.float16
        print("-> Using FP16 backbone (fast path on T4).")
    else:
        backbone.to(torch.bfloat16)
        print("-> FP16 numerics differ too much; reverted to BF16 (will be slow on T4).")
amp_dtype = model_dtype if device.type == "cuda" else torch.bfloat16
print(f"Backbone dtype={model_dtype}, amp_dtype={amp_dtype}")

# Diagnostic: which attention implementation is the backbone using? (want "sdpa" or "flash_attention_2")
print(
    "attn_implementation:",
    getattr(getattr(backbone, "config", None), "_attn_implementation", "unknown"),
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
)
peft_backbone = get_peft_model(backbone, lora_config)
peft_backbone.print_trainable_parameters()

W1012 06:11:01.072000 2212 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1012 06:11:01.145000 2212 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading base decision model vllm-sr/d3-edge...


config.json:   0%|          | 0.00/2.57k [00:00<?, ?B/s]

modeling_d3.py:   0%|          | 0.00/12.1k [00:00<?, ?B/s]

d3_runtime.py:   0%|          | 0.00/88.8k [00:00<?, ?B/s]

...

decision_config.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 31 files:   0%|          | 0/31 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/305 [00:00<?, ?it/s]

trainable params: 1,081,344 || all params: 595,402,944 || trainable%: 0.1816


In [6]:
# Cell 5: Fast Pre-tokenized Training (T4: native fp16 AMP, length bucketing, no grad checkpointing)
import gc
import random
import shutil
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

gc.collect()
torch.cuda.empty_cache()

# ----------------------------- Config -----------------------------
MAX_TOKEN_LEN = 256
TRAIN_BATCH_SIZE = 16  # was 8 with grad-accum 2 -> same effective batch (16), half the steps
GRAD_ACCUM = 1  # if you hit CUDA OOM: TRAIN_BATCH_SIZE=8, GRAD_ACCUM=2
EVAL_BATCH_SIZE = 32
BUCKET_MULT = 50  # sort by length inside windows of TRAIN_BATCH_SIZE * BUCKET_MULT samples
GRAD_CKPT = False  # 0.6B @ 256 tokens fits in 16GB; checkpointing only costs ~30% speed
epochs = 5
lr = 2e-4
# ------------------------------------------------------------------


# 1. Pre-tokenize entire dataset into memory once
def tokenize_dataset(samples, model, max_token_len=MAX_TOKEN_LEN):
    tokenized = []
    for s in samples:
        q = {"type": "choice", "instructions": s["instructions"], "criteria": s["criteria"]}
        text = model.runtime.text(s["state"], q)
        seq = model.runtime.tokenizer([text], add_special_tokens=False)["input_ids"][0]
        if len(seq) > max_token_len:
            seq = seq[-max_token_len:]
        opts = list(s["criteria"].keys())
        tokenized.append(
            {
                "seq": seq,
                "target": opts.index(s["target_choice"]),
                "count": len(opts),
            }
        )
    return tokenized


print("Pre-tokenizing dataset into RAM (one-time pass)...")
train_tok = tokenize_dataset(train_set, model)
val_tok = tokenize_dataset(val_set, model)
print(f"Pre-tokenized {len(train_tok)} train, {len(val_tok)} val samples.")

lens = np.array([len(x["seq"]) for x in train_tok])
print(
    f"Token length  p50={int(np.percentile(lens, 50))}  p95={int(np.percentile(lens, 95))}  "
    f"max={int(lens.max())}  (MAX_TOKEN_LEN={MAX_TOKEN_LEN}; lower it if p95 is far below)"
)


def collate_batch(batch, model, device):
    max_len = max(len(x["seq"]) for x in batch)
    pad_id = model.runtime.pad_id
    input_ids = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)
    targets, counts = [], []
    for i, x in enumerate(batch):
        seq = x["seq"]
        start = max_len - len(seq)  # left padding (last token is used for readout)
        input_ids[i, start:] = torch.as_tensor(seq, dtype=torch.long)
        attention_mask[i, start:] = 1
        targets.append(x["target"])
        counts.append(x["count"])
    return (
        input_ids.to(device, non_blocking=True),
        attention_mask.to(device, non_blocking=True),
        torch.tensor(targets, dtype=torch.long, device=device),
        torch.tensor(counts, dtype=torch.long, device=device),
    )


def make_bucketed_batches(data, batch_size, mult=BUCKET_MULT):
    """Shuffle, sort by length inside big windows, cut into batches, shuffle batch order.
    Batches contain similar-length samples -> far less padding, still random across epochs."""
    idx = list(range(len(data)))
    random.shuffle(idx)
    window = batch_size * mult
    batches = []
    for s in range(0, len(idx), window):
        part = sorted(idx[s : s + window], key=lambda i: len(data[i]["seq"]))
        batches += [part[j : j + batch_size] for j in range(0, len(part), batch_size)]
    random.shuffle(batches)
    return [[data[i] for i in b] for b in batches]


# Readout matrix cast ONCE (frozen, not trained)
readout_w = readout.detach().to(
    device=device, dtype=torch.float32
)  # checkpoint keeps readout in FP32


def amp_ctx():
    return torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=(device.type == "cuda"))


def compute_loss(backbone, readout_w, input_ids, attention_mask, targets, counts):
    out = backbone(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
    hidden = out.last_hidden_state[:, -1]
    # float32 logits (readout is FP32 in the checkpoint; also -1e9 would overflow to -inf in fp16)
    logits = (hidden.float() @ readout_w.T).float()
    invalid = torch.arange(logits.size(-1), device=logits.device)[None, :] >= counts[:, None]
    logits = logits.float().masked_fill(invalid, -1e4)  # vectorized (no per-sample Python loop)
    loss = F.cross_entropy(logits, targets)
    correct = (logits.argmax(dim=-1) == targets).sum()
    return loss, correct


@torch.no_grad()
def evaluate(val_sorted, model, backbone, device, batch_size=EVAL_BATCH_SIZE):
    backbone.eval()
    tot_loss = torch.zeros((), device=device)
    tot_correct = torch.zeros((), device=device)
    n = 0
    with amp_ctx():
        for i in range(0, len(val_sorted), batch_size):
            batch = val_sorted[i : i + batch_size]
            ids, mask, targets, counts = collate_batch(batch, model, device)
            loss, correct = compute_loss(backbone, readout_w, ids, mask, targets, counts)
            tot_loss += loss.detach().float() * len(batch)
            tot_correct += correct
            n += len(batch)
    return tot_loss.item() / max(1, n), tot_correct.item() / max(1, n)


# Sort val by length once (order doesn't matter for aggregate metrics, but padding does)
val_sorted = sorted(val_tok, key=lambda x: len(x["seq"]))

if GRAD_CKPT:
    peft_backbone.gradient_checkpointing_enable()
    try:
        peft_backbone.get_base_model().enable_input_require_grads()
    except Exception:
        pass

# Zero-shot baseline evaluation
zero_loss, zero_acc = evaluate(val_sorted, model, peft_backbone, device)
print(f"Zero-shot Baseline -> Val Loss: {zero_loss:.4f}, Val Acc: {zero_acc * 100:.2f}%")
print(
    f"dtype={model_dtype}, amp_dtype={amp_dtype}  (zero-shot should look sane; NaN => fp16 overflow)"
)

# Optimizer: only trainable (LoRA) params, fused CUDA kernel
trainable_params = [p for p in peft_backbone.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    trainable_params, lr=lr, weight_decay=0.01, fused=(device.type == "cuda")
)
# GradScaler is only needed for fp16; when disabled, scale/step/update are pass-throughs
scaler = torch.amp.GradScaler(
    "cuda", enabled=(device.type == "cuda" and amp_dtype == torch.float16)
)

checkpoints_dir = Path("./checkpoints")
checkpoints_dir.mkdir(parents=True, exist_ok=True)
best_adapter_dir = Path("./best_adapter")
best_adapter_dir.mkdir(parents=True, exist_ok=True)

history = {
    "epochs": [0],
    "train_loss": [float("nan")],
    "val_loss": [zero_loss],
    "val_acc": [zero_acc],
}

print(
    f"\nStarting Fast Training ({len(train_tok)} samples, batch_size={TRAIN_BATCH_SIZE}, "
    f"accum={GRAD_ACCUM}, grad_ckpt={GRAD_CKPT})..."
)
for epoch in range(1, epochs + 1):
    peft_backbone.train()
    batches = make_bucketed_batches(train_tok, TRAIN_BATCH_SIZE)
    total_train_loss = torch.zeros((), device=device)
    optimizer.zero_grad(set_to_none=True)
    t0 = time.time()

    for step, batch in enumerate(tqdm(batches, desc=f"Epoch {epoch}/{epochs}", leave=False), 1):
        ids, mask, targets, counts = collate_batch(batch, model, device)

        with amp_ctx():
            loss, _ = compute_loss(peft_backbone, readout_w, ids, mask, targets, counts)

        scaler.scale(loss / GRAD_ACCUM).backward()
        total_train_loss += loss.detach().float() * len(batch)  # no .item() -> no GPU sync per step

        if step % GRAD_ACCUM == 0 or step == len(batches):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

    avg_train_loss = total_train_loss.item() / len(train_tok)
    train_secs = time.time() - t0
    val_loss, val_acc = evaluate(val_sorted, model, peft_backbone, device)

    if not np.isfinite(avg_train_loss):
        print(
            "WARNING: non-finite train loss (fp16 overflow?). See notes: try lower lr or bf16 on A100/L4."
        )

    history["epochs"].append(epoch)
    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    epoch_dir = checkpoints_dir / f"adapter_epoch_{epoch}"
    epoch_dir.mkdir(parents=True, exist_ok=True)
    peft_backbone.save_pretrained(str(epoch_dir))

    delta_loss = val_loss - history["val_loss"][epoch - 1]
    delta_acc = (val_acc - history["val_acc"][epoch - 1]) * 100
    print(
        f"Epoch {epoch}/{epochs} | {train_secs:.0f}s ({len(train_tok) / train_secs:.1f} samples/s) | "
        f"Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} ({delta_loss:+.4f}) | "
        f"Val Acc: {val_acc * 100:.2f}% ({delta_acc:+.2f}%)"
    )

# Detect Elbow Knee Point (min val_loss over trained epochs, secondary tie-breaker max val_acc)
val_losses_trained = np.array(history["val_loss"][1:])
best_loss_epoch = int(np.argmin(val_losses_trained)) + 1
elbow_epoch = best_loss_epoch
for e in range(1, epochs + 1):
    if (
        abs(history["val_loss"][e] - history["val_loss"][elbow_epoch]) < 1e-4
        and history["val_acc"][e] > history["val_acc"][elbow_epoch]
    ):
        elbow_epoch = e

elbow_val_loss = history["val_loss"][elbow_epoch]
elbow_val_acc = history["val_acc"][elbow_epoch]

print("\n" + "=" * 60)
print("Elbow Detection Complete:")
print(f"  Selected Knee Point   : Epoch {elbow_epoch} / {epochs}")
print(f"  Elbow Validation Loss : {elbow_val_loss:.4f} (Zero-shot: {zero_loss:.4f})")
print(f"  Elbow Validation Acc  : {elbow_val_acc * 100:.2f}% (Zero-shot: {zero_acc * 100:.2f}%)")

# Robust copy to ./best_adapter
shutil.rmtree(best_adapter_dir, ignore_errors=True)
best_adapter_dir.mkdir(parents=True, exist_ok=True)
shutil.copytree(
    checkpoints_dir / f"adapter_epoch_{elbow_epoch}", best_adapter_dir, dirs_exist_ok=True
)
print(f"  Active Merging Target : {best_adapter_dir.resolve()} <= adapter_epoch_{elbow_epoch}")
print("=" * 60)

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Zero-shot Baseline -> Val Loss: 2.8229, Val Acc: 17.50%

Starting Training...
Epoch 1/5 | Train Loss: 1.6940 | Val Loss: 1.4896 | Val Acc: 37.50%
  -> Saved best adapter checkpoint (Val Acc: 37.50%)
Epoch 2/5 | Train Loss: 1.2535 | Val Loss: 1.3264 | Val Acc: 50.00%
  -> Saved best adapter checkpoint (Val Acc: 50.00%)
Epoch 3/5 | Train Loss: 0.9934 | Val Loss: 1.2793 | Val Acc: 47.50%
Epoch 4/5 | Train Loss: 0.8207 | Val Loss: 1.4676 | Val Acc: 50.00%
  -> Saved best adapter checkpoint (Val Acc: 50.00%)
Epoch 5/5 | Train Loss: 0.7330 | Val Loss: 1.3011 | Val Acc: 42.50%

Training Complete. Best Validation Accuracy: 50.00%


In [ ]:
# Cell 6: Elbow Method Visualization (Loss & Accuracy Trajectories)
from pathlib import Path

import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Cross-Entropy Loss Trajectory & Knee Detection
ax1.plot(
    history["epochs"][1:],
    history["train_loss"][1:],
    "b--o",
    label="Training Loss",
    linewidth=2,
    markersize=6,
)
ax1.plot(
    history["epochs"],
    history["val_loss"],
    "r-s",
    label="Validation Loss",
    linewidth=2,
    markersize=6,
)
ax1.axvline(
    x=elbow_epoch, color="gray", linestyle=":", linewidth=2, label=f"Elbow Epoch ({elbow_epoch})"
)

if elbow_epoch < epochs:
    ax1.axvspan(elbow_epoch, epochs, color="gray", alpha=0.12, label="Diminishing Returns Zone")

ax1.scatter(
    [elbow_epoch], [elbow_val_loss], color="red", s=140, zorder=5, edgecolors="black", linewidth=1.5
)
ax1.annotate(
    f"Elbow Knee Point\nEpoch {elbow_epoch} (Loss: {elbow_val_loss:.4f})",
    xy=(elbow_epoch, elbow_val_loss),
    xytext=(elbow_epoch + 0.15, elbow_val_loss + 0.25),
    arrowprops=dict(arrowstyle="->", color="black", lw=1.2),
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#fff2cc", edgecolor="#d6b656"),
    fontsize=9,
    fontweight="bold",
)
ax1.set_xlabel("Epoch", fontsize=11, fontweight="bold")
ax1.set_ylabel("Cross-Entropy Loss", fontsize=11, fontweight="bold")
ax1.set_title("D3 Decision Readout Loss Trajectory", fontsize=12, fontweight="bold")
ax1.set_xticks(range(0, epochs + 1))
ax1.grid(True, linestyle="--", alpha=0.6)
ax1.legend(loc="upper right", fontsize=9)

# Subplot 2: Validation Accuracy Progression
val_acc_pct = [acc * 100 for acc in history["val_acc"]]
ax2.plot(
    history["epochs"], val_acc_pct, "g-^", label="Validation Accuracy", linewidth=2, markersize=7
)
ax2.axhline(
    y=zero_acc * 100,
    color="gray",
    linestyle="--",
    alpha=0.7,
    label=f"Zero-Shot Baseline ({zero_acc * 100:.1f}%)",
)
ax2.axvline(x=elbow_epoch, color="gray", linestyle=":", linewidth=2)
ax2.scatter(
    [elbow_epoch],
    [elbow_val_acc * 100],
    color="green",
    s=140,
    zorder=5,
    edgecolors="black",
    linewidth=1.5,
)
ax2.annotate(
    f"Elbow Peak Acc\n{elbow_val_acc * 100:.2f}%",
    xy=(elbow_epoch, elbow_val_acc * 100),
    xytext=(elbow_epoch - 0.7 if elbow_epoch > 1 else elbow_epoch + 0.2, elbow_val_acc * 100 - 6),
    arrowprops=dict(arrowstyle="->", color="black", lw=1.2),
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#d5e8d4", edgecolor="#82b366"),
    fontsize=9,
    fontweight="bold",
)
ax2.set_xlabel("Epoch", fontsize=11, fontweight="bold")
ax2.set_ylabel("Validation Accuracy (%)", fontsize=11, fontweight="bold")
ax2.set_title("D3 Navigation Validation Accuracy Progression", fontsize=12, fontweight="bold")
ax2.set_xticks(range(0, epochs + 1))
ax2.grid(True, linestyle="--", alpha=0.6)
ax2.legend(loc="lower right", fontsize=9)

plt.tight_layout()
chart_path = Path("d3_elbow_curve.png")
plt.savefig(chart_path, dpi=300)
print(f"Saved elbow visualization chart to {chart_path.resolve()}")
plt.show()

In [ ]:
# Cell 7: Evaluation Comparison & Epoch Marginal Gains
print("=" * 68)
print(
    f"{'Epoch':<8}{'Train Loss':<14}{'Val Loss':<12}{'Val Acc (%)':<14}{'Delta Loss':<12}{'Status'}"
)
print("-" * 68)
for e in range(len(history["epochs"])):
    ep = history["epochs"][e]
    tr_l = f"{history['train_loss'][e]:.4f}" if e > 0 else "N/A (Zero-shot)"
    vl_l = f"{history['val_loss'][e]:.4f}"
    vl_a = f"{history['val_acc'][e] * 100:.2f}%"
    delta = f"{history['val_loss'][e] - history['val_loss'][e - 1]:+.4f}" if e > 0 else "-"
    marker = "<-- [ELBOW KNEE POINT]" if ep == elbow_epoch else ("Baseline" if ep == 0 else "")
    print(f"{ep:<8}{tr_l:<14}{vl_l:<12}{vl_a:<14}{delta:<12}{marker}")
print("=" * 68)
print(f"Zero-shot Baseline Acc : {zero_acc * 100:.2f}% (Loss: {zero_loss:.4f})")
print(
    f"Elbow Checkpoint (Ep {elbow_epoch}): {elbow_val_acc * 100:.2f}% (Loss: {elbow_val_loss:.4f})"
)
print(f"Absolute Improvement   : +{(elbow_val_acc - zero_acc) * 100:.2f}%")
print("=" * 68)

In [ ]:
# Cell 8: Merge LoRA Weights & Push Model to Hugging Face Hub
import shutil
import tempfile
from pathlib import Path

import torch
from huggingface_hub import HfApi, snapshot_download
from peft import PeftModel
from safetensors.torch import save_file
from transformers import AutoModel

HUB_MODEL_ID = "ersaasyhar/d3-edge-indo-legal"
best_adapter_dir = Path("./best_adapter")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Safety guard: ensure adapter_config.json exists in best_adapter_dir
if not (best_adapter_dir / "adapter_config.json").exists():
    cand_checkpoints = sorted(Path("./checkpoints").glob("adapter_epoch_*"))
    if cand_checkpoints:
        fallback = cand_checkpoints[0]
        shutil.copytree(fallback, best_adapter_dir, dirs_exist_ok=True)
        print(f"Restored adapter from {fallback} to {best_adapter_dir}")
    else:
        raise FileNotFoundError(
            f"No adapter checkpoints found in ./checkpoints or {best_adapter_dir}"
        )

print(f"Loading elbow-selected adapter from {best_adapter_dir} (Epoch {elbow_epoch})...")
base_model = AutoModel.from_pretrained(
    BASE_MODEL_NAME,
    trust_remote_code=True,
)
base_model = base_model.to(device)

peft_model = PeftModel.from_pretrained(
    base_model.runtime.backbone,
    str(best_adapter_dir),
)

print("Merging LoRA weights into the Qwen backbone...")
merged_backbone = peft_model.merge_and_unload()

staging_dir = Path(tempfile.mkdtemp(prefix="d3_colab_merge_"))
try:
    print(f"Downloading files from {BASE_MODEL_NAME}...")
    base_files_dir = snapshot_download(BASE_MODEL_NAME)

    # Copy base scaffold to staging directory
    shutil.copytree(base_files_dir, staging_dir, dirs_exist_ok=True)

    # Delete stale manifest so verify_package cleanly returns None
    manifest_file = staging_dir / "MODEL_MANIFEST.json"
    if manifest_file.exists():
        manifest_file.unlink()

    # Delete any compiled pycache artifacts
    for pycache in staging_dir.rglob("__pycache__"):
        shutil.rmtree(pycache, ignore_errors=True)

    print("Overwriting backbone weights with fine-tuned merged weights...")
    weight_file = staging_dir / "model-00001-of-00001.safetensors"
    if not weight_file.exists():
        weight_file = staging_dir / "model.safetensors"
    save_file({k: v.cpu() for k, v in merged_backbone.state_dict().items()}, weight_file)

    # Copy elbow curve visualization if available
    chart_file = Path("d3_elbow_curve.png")
    if chart_file.exists():
        shutil.copy2(chart_file, staging_dir / "d3_elbow_curve.png")
        print("Copied d3_elbow_curve.png to repository staging directory.")

    print("Generating Hugging Face Model Card README.md...")
    readme_content = f"""---
base_model: {BASE_MODEL_NAME}
language:
- id
license: apache-2.0
tags:
- zero-shot-classification
- text-classification
- decision-model
- system-one
- indonesian
- legal
- rag
- d3-edge
pipeline_tag: zero-shot-classification
---

# {HUB_MODEL_ID}

Fine-tuned decision model based on [{BASE_MODEL_NAME}](https://huggingface.co/{BASE_MODEL_NAME}) (0.6B) specialized for Indonesian statutory tree navigation trajectories (`root_to_bab` and `bab_to_pasal`).

## Model Details
- **Base Architecture**: Qwen3.5 0.6B with 255-way linear readout (`readout.safetensors`).
- **Domain**: Indonesian legal tree navigation (Undang-Undang, Peraturan Pemerintah).
- **Optimization**: 5-Epoch Elbow Method (Optimal Knee Point: Epoch {elbow_epoch}).
- **Validation Accuracy**: {elbow_val_acc * 100:.2f}% (Baseline: {zero_acc * 100:.2f}%).
- **Validation Loss**: {elbow_val_loss:.4f} (Baseline: {zero_loss:.4f}).
- **Training Trajectories**: [`ersaasyhar/d3-indo-legal-navigation`](https://huggingface.co/datasets/ersaasyhar/d3-indo-legal-navigation).

## Elbow Method Convergence Curve
![D3 Elbow Curve](d3_elbow_curve.png)

## Usage with D3Navigator

```python
from src.tree_reasoning.d3_navigator import D3Navigator

navigator = D3Navigator(model_name="{HUB_MODEL_ID}")
results = navigator.retrieve("Apa syarat pendirian perseroan terbatas?", top_k=5)
for chunk in results:
    print(f"{{chunk.doc_id}} - {{chunk.title}} (p. {{chunk.page_number}})")
```
"""
    (staging_dir / "README.md").write_text(readme_content, encoding="utf-8")

    print("Sanity-checking merged model loading from staging directory...")
    test_model = AutoModel.from_pretrained(str(staging_dir), trust_remote_code=True)
    test_model = test_model.to(device)
    assert test_model is not None, "Merged model failed to load correctly"
    print("Sanity check passed!")

    print(f"Pushing merged model repository to Hugging Face Hub: {HUB_MODEL_ID}...")
    api = HfApi()
    api.create_repo(repo_id=HUB_MODEL_ID, repo_type="model", exist_ok=True)
    api.upload_folder(
        folder_path=str(staging_dir),
        repo_id=HUB_MODEL_ID,
        repo_type="model",
        commit_message=f"Fine-tune d3-edge on Indo legal navigation via elbow method (Epoch {elbow_epoch}, Val Acc: {elbow_val_acc * 100:.1f}%)",
    )
    print(f"Successfully uploaded model to: https://huggingface.co/{HUB_MODEL_ID}")
finally:
    shutil.rmtree(staging_dir, ignore_errors=True)